# UFC Gold Layer

Builds the analytics-ready star schema in `ufc_gold` — `fact_fights`, `dim_fighter`, `dim_event`, and `dim_time` — from the silver tables, then validates row counts across all three layers. Depends on the **02_silver_layer** notebook.

In [0]:
# ============================================================
# GOLD LAYER — Setup
# ============================================================
GOLD_SCHEMA = "ufc_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"COMMENT ON SCHEMA {GOLD_SCHEMA} IS 'Gold layer: analytics-ready UFC fact and dimension tables'")

def silver_table_exists(table_name):
    try:
        rows = spark.sql("SHOW TABLES IN ufc_silver").collect()
        return any(r.tableName == table_name for r in rows)
    except Exception:
        return False

from pyspark.sql.functions import *

print("Gold schema ready. Silver tables available:")
display(spark.sql("SHOW TABLES IN ufc_silver"))

In [0]:
# --- Gold: fact_fights (enriched fights with event details) ---
if silver_table_exists("fights"):
    print("=== Building fact_fights ===")
    df_fights = spark.read.table("ufc_silver.fights")
    if silver_table_exists("events"):
        df_events = spark.read.table("ufc_silver.events")
        if "event_name" in df_fights.columns and "event" in df_events.columns:
            df_events_clean = df_events.select(
                col("event"),
                col("date").alias("event_date"),
                col("location").alias("event_location"),
                col("city").alias("event_city"),
                col("state").alias("event_state"),
                col("country").alias("event_country"),
                col("event_year"), col("event_month"), col("event_quarter")
            )
            df_fights = (df_fights.join(df_events_clean, on=col("event_name") == col("event"), how="left"))
            print("  Joined with events on event_name = event")
    df_fights = df_fights.withColumn("fight_id", monotonically_increasing_id())
    df_fights.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.fact_fights")
    print(f"  Rows: {df_fights.count()}")
    display(df_fights.select("fight_id", "event_name", "r_fighter", "b_fighter", "winner_name", "finish_category", "method", "finish_round", "time_sec", "event_date", "event_location").limit(5))
    print(f"  ✅ {GOLD_SCHEMA}.fact_fights created!")
else:
    print("⚠️  ufc_silver.fights not found")

In [0]:
# --- Gold: dim_fighter ---
if silver_table_exists("fighters"):
    print("=== Building dim_fighter ===")
    df_dim = spark.read.table("ufc_silver.fighters").withColumn("fighter_key", monotonically_increasing_id())
    df_dim.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.dim_fighter")
    print(f"  Rows: {df_dim.count()}")
    display(df_dim.select("fighter_key", "name", "wins", "losses", "height_cm", "weight_kg", "reach_cm", "stance", "age", "bmi", "win_rate", "weight_class").limit(5))
    print(f"  ✅ {GOLD_SCHEMA}.dim_fighter created!")
else:
    print("⚠️  ufc_silver.fighters not found")

In [0]:
# --- Gold: dim_event ---
if silver_table_exists("events"):
    print("=== Building dim_event ===")
    df_dim = spark.read.table("ufc_silver.events").withColumn("event_key", monotonically_increasing_id())
    df_dim.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.dim_event")
    print(f"  Rows: {df_dim.count()}")
    display(df_dim.select("event_key", "event", "date", "location", "city", "state", "country", "event_year", "event_month", "event_quarter").limit(5))
    print(f"  ✅ {GOLD_SCHEMA}.dim_event created!")
else:
    print("⚠️  ufc_silver.events not found")

# --- Gold: dim_time ---
if silver_table_exists("events"):
    print("\n=== Building dim_time ===")
    df_events = spark.read.table("ufc_silver.events")
    if "date" in df_events.columns:
        df_time = (df_events
            .select("date")
            .filter(col("date").isNotNull())
            .distinct()
            .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
            .withColumn("year", year(col("date")))
            .withColumn("month", month(col("date")))
            .withColumn("quarter", quarter(col("date")))
            .withColumn("day_of_week", dayofweek(col("date")))
            .withColumn("day_name", date_format(col("date"), "EEEE"))
            .withColumn("month_name", date_format(col("date"), "MMMM"))
            .withColumn("is_weekend", when(dayofweek(col("date")).isin([1, 7]), True).otherwise(False))
            .withColumnRenamed("date", "full_date"))
        df_time.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.dim_time")
        print(f"  Rows: {df_time.count()}")
        display(df_time.limit(10))
        print(f"  ✅ {GOLD_SCHEMA}.dim_time created!")
else:
    print("⚠️  ufc_silver.events not found")

In [0]:
# ============================================================
# VALIDATION — Row Counts & Table Summary
# ============================================================
print("=" * 60)
print("MEDALLION ARCHITECTURE SUMMARY")
print("=" * 60)

schemas = {
    "ufc_bronze": "Raw CSV Data",
    "ufc_silver": "Cleaned & Standardized",
    "ufc_gold": "Analytics-Ready (Star Schema)",
}

for schema, desc in schemas.items():
    print(f"\n📁 {schema.upper()} — {desc}")
    print("-" * 40)
    try:
        tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
        if not tables:
            print("  (no tables)")
        for row in tables:
            tbl = row.tableName
            try:
                cnt = spark.table(f"{schema}.{tbl}").count()
                print(f"  {tbl}: {cnt:,} rows")
            except Exception:
                print(f"  {tbl}: (error reading)")
    except Exception:
        print("  (schema not found)")

print("\n" + "=" * 60)
print("✅ Medallion architecture build complete!")
print("=" * 60)